# Demo Reset

**Purpose:** Drop all objects created during the demo so notebooks can be re-run cleanly  
**Safe to run:** This only drops demo-generated objects, not base infrastructure (stages, file formats, Cortex Search, Streamlit app)

In [ ]:
USE DATABASE CDSB_DEMO;
USE SCHEMA RAW;
USE WAREHOUSE COMPUTE_WH;

## 1. Drop Anomaly Detection Objects (Notebook 01)

In [ ]:
DROP SNOWFLAKE.ML.ANOMALY_DETECTION IF EXISTS tmr_anomaly_model;
DROP TABLE IF EXISTS TMR_ANOMALY_RESULTS;
DROP VIEW IF EXISTS TMR_TRAINING_DATA;
DROP VIEW IF EXISTS TMR_TEST_DATA;
DROP VIEW IF EXISTS TMR_MONTHLY_REGISTRATIONS;
DROP VIEW IF EXISTS TMR_REGIONAL_REGISTRATIONS;
DROP TABLE IF EXISTS TMR_SUBURB_MONTHLY;
DROP TABLE IF EXISTS TMR_REGISTRATIONS;

## 2. Drop Process Mining Objects (Notebook 02)

In [ ]:
DROP TABLE IF EXISTS HANDLED_BY_RELS;
DROP TABLE IF EXISTS IN_REGION_RELS;
DROP TABLE IF EXISTS PROCESS_RELATIONSHIPS;
DROP TABLE IF EXISTS CASE_NODES;
DROP TABLE IF EXISTS HANDLER_NODES;
DROP TABLE IF EXISTS REGION_NODES;
DROP TABLE IF EXISTS PROCESS_NODES;
DROP TABLE IF EXISTS SERVICE_REQUEST_EVENTS;

## 3. Drop Neo4j Graph Analytics Results (Notebook 03)

In [ ]:
DROP VIEW IF EXISTS VIZ_CASE_SAMPLE;
DROP VIEW IF EXISTS VIZ_HANDLED_BY;
DROP VIEW IF EXISTS VIZ_IN_REGION;
DROP VIEW IF EXISTS VIZ_HANDLER_PR;
DROP VIEW IF EXISTS VIZ_REGION_PR;
DROP VIEW IF EXISTS VIZ_HANDLED_BY_SAMPLE;
DROP VIEW IF EXISTS VIZ_IN_REGION_SAMPLE;
DROP VIEW IF EXISTS VIZ_CASE_LOUVAIN;
DROP VIEW IF EXISTS VIZ_HANDLER_LOUVAIN;
DROP VIEW IF EXISTS VIZ_REGION_LOUVAIN;
DROP TABLE IF EXISTS CASE_WCC_RESULTS;
DROP TABLE IF EXISTS HANDLER_WCC_RESULTS;
DROP TABLE IF EXISTS REGION_WCC_RESULTS;
DROP TABLE IF EXISTS HANDLER_PAGERANK_RESULTS;
DROP TABLE IF EXISTS REGION_PAGERANK_RESULTS;
DROP TABLE IF EXISTS CASE_LOUVAIN_RESULTS;
DROP TABLE IF EXISTS HANDLER_LOUVAIN_RESULTS;
DROP TABLE IF EXISTS REGION_LOUVAIN_RESULTS;

## 4. Drop ML Model Quality Objects (Notebook 05)

In [ ]:
DROP SNOWFLAKE.ML.ANOMALY_DETECTION IF EXISTS tmr_anomaly_model_clean;
DROP TABLE IF EXISTS TMR_ANOMALY_RESULTS_CLEAN;
DROP VIEW IF EXISTS TMR_TRAINING_DATA_CLEAN;

## 5. Drop Audio/Video Pipeline Objects (Notebook 06)

In [ ]:
DROP TABLE IF EXISTS CALL_ANALYTICS;
DROP TABLE IF EXISTS CALL_SENTIMENT;
DROP TABLE IF EXISTS CALL_REDACTED;
DROP TABLE IF EXISTS CALL_TRANSLATIONS;
DROP TABLE IF EXISTS CALL_TRANSCRIPTS;
DROP TABLE IF EXISTS CALL_SCENARIOS;

## 6. Verify Clean State
Only base infrastructure tables should remain

In [ ]:
SELECT TABLE_NAME, TABLE_TYPE
FROM INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'RAW'
  AND TABLE_NAME IN (
      'TMR_REGISTRATIONS','TMR_SUBURB_MONTHLY','TMR_ANOMALY_RESULTS',
      'TMR_TRAINING_DATA','TMR_TEST_DATA','TMR_MONTHLY_REGISTRATIONS','TMR_REGIONAL_REGISTRATIONS',
      'SERVICE_REQUEST_EVENTS','PROCESS_NODES','PROCESS_RELATIONSHIPS',
      'CASE_NODES','HANDLER_NODES','REGION_NODES',
      'HANDLED_BY_RELS','IN_REGION_RELS',
      'CASE_WCC_RESULTS','HANDLER_WCC_RESULTS','REGION_WCC_RESULTS',
      'HANDLER_PAGERANK_RESULTS','REGION_PAGERANK_RESULTS',
      'CASE_LOUVAIN_RESULTS','HANDLER_LOUVAIN_RESULTS','REGION_LOUVAIN_RESULTS',
      'VIZ_CASE_SAMPLE','VIZ_HANDLED_BY','VIZ_IN_REGION',
      'VIZ_HANDLER_PR','VIZ_REGION_PR','VIZ_HANDLED_BY_SAMPLE','VIZ_IN_REGION_SAMPLE',
      'VIZ_CASE_LOUVAIN','VIZ_HANDLER_LOUVAIN','VIZ_REGION_LOUVAIN',
      'TMR_ANOMALY_RESULTS_CLEAN','TMR_TRAINING_DATA_CLEAN',
      'CALL_ANALYTICS','CALL_SENTIMENT','CALL_REDACTED',
      'CALL_TRANSLATIONS','CALL_TRANSCRIPTS','CALL_SCENARIOS'
  )
ORDER BY TABLE_NAME;

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
remaining = session.sql("""
    SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES
    WHERE TABLE_SCHEMA = 'RAW'
      AND TABLE_NAME IN ('TMR_REGISTRATIONS','TMR_SUBURB_MONTHLY','TMR_ANOMALY_RESULTS',
          'SERVICE_REQUEST_EVENTS','PROCESS_NODES','PROCESS_RELATIONSHIPS',
          'CASE_NODES','HANDLER_NODES','REGION_NODES',
          'HANDLED_BY_RELS','IN_REGION_RELS',
          'CASE_WCC_RESULTS','HANDLER_WCC_RESULTS','REGION_WCC_RESULTS',
          'HANDLER_PAGERANK_RESULTS','REGION_PAGERANK_RESULTS',
          'CASE_LOUVAIN_RESULTS','HANDLER_LOUVAIN_RESULTS','REGION_LOUVAIN_RESULTS',
          'VIZ_CASE_SAMPLE','VIZ_HANDLED_BY','VIZ_IN_REGION',
          'VIZ_HANDLER_PR','VIZ_REGION_PR','VIZ_HANDLED_BY_SAMPLE','VIZ_IN_REGION_SAMPLE',
          'VIZ_CASE_LOUVAIN','VIZ_HANDLER_LOUVAIN','VIZ_REGION_LOUVAIN',
          'TMR_ANOMALY_RESULTS_CLEAN','TMR_TRAINING_DATA_CLEAN',
          'CALL_ANALYTICS','CALL_SENTIMENT','CALL_REDACTED',
          'CALL_TRANSLATIONS','CALL_TRANSCRIPTS','CALL_SCENARIOS')
""").to_pandas()
if len(remaining) == 0:
    print('All demo objects cleaned up successfully!')
else:
    print(f'WARNING: {len(remaining)} objects still remain:')
    print(remaining.to_string(index=False))

## Done

Demo environment is reset. You can now re-run the notebooks in order:
1. **Anomaly Detection** - loads data + trains model
2. **Process Mining** - generates events + analyses
3. **Neo4j Graph Analytics** - runs graph algorithms
4. **ML Model Quality** - false positive elimination (requires NB01 first)
5. **Audio/Video AI Pipeline** - transcribe, translate, redact, sentiment